# 🧠 Nero AI — GPU Training on Google Colab
**Model:** BiologicLLMV2 (~200M params) with sliding window attention + plastic synapses

**Steps:**
1. Mount Google Drive (to save checkpoints)
2. Upload your `neuro/` folder or clone from GitHub
3. Install dependencies
4. Train on a real text corpus
5. Save trained model to Drive

**Runtime:** Set to `GPU → T4` (Runtime → Change runtime type → GPU)

In [ ]:
# ── STEP 1: Mount Google Drive ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

In [ ]:
# ── STEP 2: Upload neuro code ────────────────────────────────────
# Option A (recommended): upload the neuro folder as a zip from your PC
import os

if not os.path.exists('/content/neuro'):
    from google.colab import files
    print('Upload your neuro folder as a ZIP file (zip the entire neuro/ directory)')
    uploaded = files.upload()  # upload neuro.zip
    zip_name = list(uploaded.keys())[0]
    os.makedirs('/content/neuro', exist_ok=True)
    import zipfile
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content/')
    # Handle common zip layouts
    if not os.path.exists('/content/neuro/biologic_v2.py'):
        # Files may have extracted into a subfolder
        for root, dirs, fls in os.walk('/content'):
            if 'biologic_v2.py' in fls:
                import shutil
                shutil.copytree(root, '/content/neuro', dirs_exist_ok=True)
                break
    print('Neuro code uploaded!')
else:
    print('neuro/ already present')

os.chdir('/content/neuro')
print('Working dir:', os.getcwd())
print('Files:', [f for f in os.listdir('.') if f.endswith('.py')])

In [ ]:
# ── STEP 3: Install dependencies ────────────────────────────────
!pip install -q gradio fastapi uvicorn datasets
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── STEP 4: Build tokenizer (if bpe_vocab.json not included) ────
import sys, os
sys.path.insert(0, '/content/neuro')
os.chdir('/content/neuro')

from tokenizer import BPETokenizer

if not os.path.exists('bpe_vocab.json'):
    print('Training BPE tokenizer from seed data...')
    from biologic_v2 import SEED_TEXTS
    all_seed = '\n'.join(SEED_TEXTS.values())
    tok = BPETokenizer(vocab_size=4096)
    tok.train(all_seed)
    tok.save('bpe_vocab.json')
    print(f'Tokenizer saved: {tok.get_vocab_size()} tokens')
else:
    tok = BPETokenizer(vocab_size=4096)
    tok.load('bpe_vocab.json')
    print(f'Tokenizer loaded: {tok.get_vocab_size()} tokens')

In [ ]:
# ── STEP 5: Download training corpus (Project Gutenberg) ─────────
# We download several classic books as plain text — free, high quality English
import urllib.request, os

GUTENBERG_BOOKS = [
    ('pg1342.txt', 'https://www.gutenberg.org/files/1342/1342-0.txt'),   # Pride and Prejudice
    ('pg11.txt',   'https://www.gutenberg.org/files/11/11-0.txt'),       # Alice in Wonderland
    ('pg1661.txt', 'https://www.gutenberg.org/files/1661/1661-0.txt'),   # Sherlock Holmes
    ('pg2701.txt', 'https://www.gutenberg.org/files/2701/2701-0.txt'),   # Moby Dick
    ('pg84.txt',   'https://www.gutenberg.org/files/84/84-0.txt'),       # Frankenstein
    ('pg1232.txt', 'https://www.gutenberg.org/files/1232/1232-0.txt'),   # The Prince (philosophy)
    ('pg2554.txt', 'https://www.gutenberg.org/files/2554/2554-0.txt'),   # Crime and Punishment
    ('pg5200.txt', 'https://www.gutenberg.org/files/5200/5200-0.txt'),   # Metamorphosis
]

corpus_dir = '/content/corpus_books'
os.makedirs(corpus_dir, exist_ok=True)

for fname, url in GUTENBERG_BOOKS:
    fpath = os.path.join(corpus_dir, fname)
    if not os.path.exists(fpath):
        try:
            urllib.request.urlretrieve(url, fpath)
            size = os.path.getsize(fpath) / 1e3
            print(f'  Downloaded {fname} ({size:.0f} KB)')
        except Exception as e:
            print(f'  Failed {fname}: {e}')
    else:
        print(f'  Already have {fname}')

# Combine into single corpus file
corpus_path = '/content/neuro/corpus.txt'
with open(corpus_path, 'w', encoding='utf-8') as out:
    for fname, _ in GUTENBERG_BOOKS:
        fpath = os.path.join(corpus_dir, fname)
        if os.path.exists(fpath):
            with open(fpath, 'r', encoding='utf-8', errors='replace') as f:
                out.write(f.read())
            out.write('\n\n')

size_mb = os.path.getsize(corpus_path) / 1e6
print(f'\nCorpus assembled: {size_mb:.1f} MB at {corpus_path}')

In [ ]:
# ── STEP 6: Create model ─────────────────────────────────────────
import torch, sys, os
sys.path.insert(0, '/content/neuro')
os.chdir('/content/neuro')

from tokenizer import BPETokenizer
from biologic_v2 import BiologicLLMV2, SEED_TEXTS, DEVICE

tokenizer = BPETokenizer(vocab_size=4096)
tokenizer.load('bpe_vocab.json')
print(f'Tokenizer: {tokenizer.get_vocab_size()} tokens')

# Auto-pick model size based on available VRAM
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb >= 14:       # A100/V100
        embed_dim, num_layers = 1408, 12
        label = '300M'
    elif vram_gb >= 10:     # T4 16GB
        embed_dim, num_layers = 1408, 8
        label = '200M'
    else:                   # T4 <12GB (shared)
        embed_dim, num_layers = 688, 8
        label = '50M'
    print(f'VRAM: {vram_gb:.1f} GB → using {label} config (embed={embed_dim}, layers={num_layers})')
else:
    embed_dim, num_layers = 256, 4
    label = '5M (CPU)'

model = BiologicLLMV2(
    vocab_size=tokenizer.vocab_size,
    embed_dim=embed_dim, num_heads=8, num_layers=num_layers,
    max_context=16384, window_size=1024, dropout=0.1,
    device=DEVICE
)
model.growth_enabled = False
model.hebbian_enabled = False  # disable during bulk training for speed
model.eos_token_id = tokenizer.SPECIAL_TOKENS.get('<EOS>', 3)
model.bos_token_id = tokenizer.SPECIAL_TOKENS.get('<BOS>', 2)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model: {total_params:,} params ({total_params/1e6:.0f}M) on {DEVICE}')

# Try to load existing checkpoint from Drive
DRIVE_CHECKPOINT = '/content/drive/MyDrive/nero_v1.pt'
if os.path.exists(DRIVE_CHECKPOINT):
    print(f'Loading checkpoint from Drive: {DRIVE_CHECKPOINT}')
    sd = torch.load(DRIVE_CHECKPOINT, map_location=DEVICE, weights_only=True)
    try:
        model.load_state_dict(sd)
        print('Checkpoint loaded — continuing training from saved weights')
    except RuntimeError as e:
        print(f'Checkpoint dim mismatch — starting fresh. Detail: {e}')
else:
    print('No Drive checkpoint found — starting from scratch')

In [ ]:
# ── STEP 7: Seed training (fast warm-up on built-in knowledge) ───
import torch, numpy as np, time

print('Seed training (1 epoch, AMP)...')
model.train()
if model._optimizer is None:
    model._create_optimizer()

scaler = torch.amp.GradScaler(device=DEVICE.type)
seed_steps = 0
seed_loss = []

for domain, text in SEED_TEXTS.items():
    encoded = tokenizer.encode(text)
    if len(encoded) < 4:
        continue
    for i in range(0, len(encoded) - 256 - 1, 128):
        chunk = encoded[i:i+256]
        target = encoded[i+1:i+257]
        if len(chunk) != len(target) or len(chunk) < 2:
            continue
        inp = torch.tensor([chunk], dtype=torch.long, device=DEVICE)
        tgt = torch.tensor([target], dtype=torch.long, device=DEVICE)
        with torch.amp.autocast(DEVICE.type):
            _, loss, _ = model(inp, targets=tgt, return_value=False)
        if loss is not None and not (torch.isnan(loss) or torch.isinf(loss)):
            model._optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(model._optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(model._optimizer)
            scaler.update()
            seed_loss.append(loss.item())
            seed_steps += 1

if torch.cuda.is_available():
    torch.cuda.empty_cache()
avg = np.mean(seed_loss) if seed_loss else 0
print(f'Seed done: {seed_steps} steps, avg loss = {avg:.4f}')

In [ ]:
# ── STEP 8: Main corpus training ────────────────────────────────
import torch, numpy as np, random, time

CORPUS_FILE = '/content/neuro/corpus.txt'
EPOCHS      = 3          # increase to 5-10 for better quality (costs time)
CHUNK_SIZE  = 1024       # tokens per chunk
STRIDE      = 512        # overlap between chunks
BATCH_SIZE  = 8          # reduce to 4 if OOM
SAVE_EVERY  = 1          # save checkpoint to Drive every N epochs

def prepare_chunks(encoded, chunk_size=1024, stride=512):
    chunks = []
    for i in range(0, len(encoded) - chunk_size - 1, stride):
        inp = np.array(encoded[i:i+chunk_size], dtype=np.int64)
        tgt = np.array(encoded[i+1:i+chunk_size+1], dtype=np.int64)
        if len(inp) == len(tgt):
            chunks.append((inp, tgt))
    return chunks

print(f'Reading corpus: {CORPUS_FILE}')
with open(CORPUS_FILE, 'r', encoding='utf-8', errors='replace') as f:
    text = f.read()
print(f'  {len(text):,} chars | Encoding...')
encoded = tokenizer.encode(text)
print(f'  {len(encoded):,} tokens | Building chunks...')
chunks = prepare_chunks(encoded, CHUNK_SIZE, STRIDE)
print(f'  {len(chunks):,} chunks | Starting training...')
del text, encoded  # free RAM

model.train()
scaler = torch.amp.GradScaler(device=DEVICE.type)

for epoch in range(EPOCHS):
    indices = list(range(len(chunks)))
    random.shuffle(indices)
    epoch_loss = []
    t_epoch = time.time()
    
    for bidx in range(0, len(chunks), BATCH_SIZE):
        batch_idx = indices[bidx:bidx+BATCH_SIZE]
        inp_np = np.stack([chunks[i][0] for i in batch_idx])
        tgt_np = np.stack([chunks[i][1] for i in batch_idx])
        inp = torch.from_numpy(inp_np).long().to(DEVICE, non_blocking=True)
        tgt = torch.from_numpy(tgt_np).long().to(DEVICE, non_blocking=True)
        
        with torch.amp.autocast(DEVICE.type):
            logits, loss, _ = model(inp, targets=tgt, return_value=False)
        
        if loss is not None and not (torch.isnan(loss) or torch.isinf(loss)):
            model._optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(model._optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(model._optimizer)
            scaler.update()
            epoch_loss.append(loss.item())
        
        if (bidx // BATCH_SIZE) % 100 == 0 and bidx > 0:
            pct   = bidx * 100 / len(chunks)
            avg_l = np.mean(epoch_loss[-100:]) if epoch_loss else 0
            elapsed = time.time() - t_epoch
            rate  = bidx / elapsed if elapsed > 0 else 0
            eta   = (len(chunks) - bidx) / rate if rate > 0 else 0
            vram  = torch.cuda.max_memory_allocated(DEVICE) / 1e9 if torch.cuda.is_available() else 0
            print(f'  E{epoch+1} {pct:.0f}% | loss={avg_l:.4f} | {rate:.0f} ch/s | ETA {eta:.0f}s | VRAM {vram:.1f}GB', flush=True)
    
    avg = np.mean(epoch_loss) if epoch_loss else 0
    elapsed = time.time() - t_epoch
    print(f'Epoch {epoch+1}/{EPOCHS} done in {elapsed:.0f}s | avg_loss={avg:.4f}')
    
    # Save checkpoint to Drive after every epoch
    if (epoch + 1) % SAVE_EVERY == 0:
        ckpt_path = f'/content/drive/MyDrive/nero_epoch{epoch+1}.pt'
        torch.save(model.state_dict(), ckpt_path)
        sz = os.path.getsize(ckpt_path) / 1e6
        print(f'  Checkpoint saved → {ckpt_path} ({sz:.0f} MB)')

print('Training complete!')

In [ ]:
# ── STEP 9: Save final checkpoint to Drive ───────────────────────
import os, torch

final_path = '/content/drive/MyDrive/nero_v1_trained.pt'
torch.save(model.state_dict(), final_path)
sz = os.path.getsize(final_path) / 1e6
print(f'Final checkpoint saved: {final_path} ({sz:.0f} MB)')

# Also save bpe_vocab.json to Drive if not already there
import shutil
shutil.copy('/content/neuro/bpe_vocab.json', '/content/drive/MyDrive/bpe_vocab.json')
print('bpe_vocab.json saved to Drive')

In [ ]:
# ── STEP 10: Test generation ──────────────────────────────────────
model.eval()

test_prompts = [
    'Who are you?',
    'What is consciousness?',
    'Tell me about yourself.',
    'What do you think about?',
]

print('Generation tests:\n')
for prompt in test_prompts:
    prompt_ids = tokenizer.encode(f'User: {prompt}\n')
    gen_ids = model.generate(
        prompt_ids,
        max_new_tokens=100,
        temperature=0.7,
        top_k=50,
        repetition_penalty=1.1
    )
    output = tokenizer.decode(gen_ids)
    # Strip the prompt from the output
    if f'User: {prompt}' in output:
        output = output[output.index(f'User: {prompt}') + len(f'User: {prompt}'):].strip()
    print(f'Prompt: "{prompt}"')
    print(f'Output: {output[:300]}')
    print('---')

In [ ]:
# ── OPTIONAL: Download checkpoint directly to your PC ────────────
# Run this cell to get a download link for the trained model
from google.colab import files

# Download the trained model (may take a few minutes for large files)
print('Preparing download...')
files.download('/content/drive/MyDrive/nero_v1_trained.pt')

In [ ]:
# ── OPTIONAL: HuggingFace dataset training (higher quality) ──────
# Uncomment and run this instead of Step 8 for much better conversational quality
# Uses nvidia/Nemotron-Cascade-2-SFT-Data (conversation pairs)

# !pip install -q datasets
# from datasets import load_dataset
# import torch, time, numpy as np
#
# CHUNK_SIZE = 48
# STRIDE = 24
# configs = ["chat", "conversational_agent"]
# total_steps = 0
#
# model.train()
# scaler = torch.amp.GradScaler(device=DEVICE.type)
#
# for cfg in configs:
#     print(f'Loading {cfg}...')
#     ds = load_dataset("nvidia/Nemotron-Cascade-2-SFT-Data", cfg, split="train", streaming=True)
#     for i, ex in enumerate(ds):
#         msgs = ex.get('messages', [])
#         text = ' '.join(m.get('content','') for m in msgs if isinstance(m.get('content',''), str))
#         if len(text) < 20: continue
#         encoded = tokenizer.encode(text)
#         if len(encoded) < CHUNK_SIZE + 2: continue
#         for sp in range(0, len(encoded) - CHUNK_SIZE - 1, STRIDE):
#             ck = encoded[sp:sp+CHUNK_SIZE]
#             tg = encoded[sp+1:sp+CHUNK_SIZE+1]
#             inp = torch.tensor([ck], dtype=torch.long, device=DEVICE)
#             tgt2 = torch.tensor([tg], dtype=torch.long, device=DEVICE)
#             with torch.amp.autocast(DEVICE.type):
#                 _, loss, _ = model(inp, targets=tgt2, return_value=False)
#             if loss is not None and not torch.isnan(loss):
#                 model._optimizer.zero_grad(set_to_none=True)
#                 scaler.scale(loss).backward()
#                 scaler.unscale_(model._optimizer)
#                 torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
#                 scaler.step(model._optimizer)
#                 scaler.update()
#                 total_steps += 1
#         if i % 200 == 0:
#             print(f'  {cfg}: {i} examples, {total_steps} steps')
#         if i >= 5000:  # limit per config for free Colab
#             break
# print(f'Dataset training done: {total_steps} steps')